In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

In [2]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative

In [3]:
max_words = 1000
max_len = 15

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

print("Example padded sequence:\n", padded_sequences[6])

Example padded sequence:
 [ 2  6  3  4  7 20  5 21  0  0  0  0  0  0  0]


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42
)

In [5]:
embedding_dim = 50

lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
lstm_history = lstm_model.fit(
    X_train, np.array(y_train),
    epochs=20,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 218ms/step - accuracy: 0.5000 - loss: 0.6957 - val_accuracy: 0.0000e+00 - val_loss: 0.7364
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.8333 - loss: 0.6680 - val_accuracy: 0.0000e+00 - val_loss: 0.7692
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.8333 - loss: 0.6362 - val_accuracy: 0.0000e+00 - val_loss: 0.8186
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.8333 - loss: 0.6335 - val_accuracy: 0.0000e+00 - val_loss: 0.8675
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8333 - loss: 0.5824 - val_accuracy: 0.0000e+00 - val_loss: 0.9406
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.8333 - loss: 0.5782 - val_accuracy: 0.0000e+00 - val_loss: 1.0374
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.8333 - loss: 0.5525 - val_accuracy: 0.0000e+00 - val_loss: 1.2129
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.8333 - loss: 0.4650 - val_accurac

In [7]:
loss, accuracy = lstm_model.evaluate(X_test, np.array(y_test))
#print(f"Test Accuracy: {accuracy*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step - accuracy: 0.0000e+00 - loss: 1.9531


In [8]:
new_texts = [
    "I hate this movie but the ending is good",
    "Absolutely loved the story and characters"
]

seq = tokenizer.texts_to_sequences(new_texts)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
predictions = lstm_model.predict(padded_seq)

for text, pred in zip(new_texts, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"Text: {text}\nPredicted sentiment: {sentiment}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
Text: I hate this movie but the ending is good
Predicted sentiment: Positive

Text: Absolutely loved the story and characters
Predicted sentiment: Positive

